# Demo 03 : Served agents

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=500, max_retries=0)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
NOTEBOOK_DIR = ROOT / "notebooks" / "06-agent-communication" if ROOT else Path.cwd()


def stock(part="CE-CF-02"):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?", (part,)).fetchone()
    return dict(row) if row else {"part_no": part, "error": "unknown part"}


def guidance(error="F-CE-110"):
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error + " |"))


FACTS = stock()
QUESTION = "Report CE-CF-02 stock and location, then the first action and replacement condition for F-CE-110."
LIMIT = {"recursion_limit": 16}


def finish_metrics(name):
    row = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "2")),
           "calls": counter.calls, "input_tokens": counter.input_tokens,
           "output_tokens": counter.output_tokens, "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", row)
    (NOTEBOOK_DIR / (name + "-metrics.json")).write_text(json.dumps(row, indent=2) + "\n", encoding="utf-8")

## a · calling-a-served-agent

We run a local Agent Server from the installed server group. We first check a graph that has no model or .env loader with both LangSmith key names removed from the subprocess environment.

In [3]:
import atexit
import re
import subprocess
from uuid import UUID, uuid4, uuid5
import httpx

servers = []


def stop_servers():
    for process, log, handle in reversed(servers):
        if process.poll() is None:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"], capture_output=True, check=False)
            else:
                os.killpg(process.pid, 15)
            process.wait(timeout=20)
        handle.close()
    servers.clear()


atexit.register(stop_servers)


NAMESPACE_GRAPH = UUID("6ba7b821-9dad-11d1-80b4-00c04fd430c8")


def start_server(port, graphs, folder, keyless=False):
    if port not in {2124, 2125}:
        raise ValueError("Use the module's reserved ports.")
    folder.mkdir(parents=True, exist_ok=True)
    config = folder / "langgraph.json"
    config.write_text(json.dumps({"dependencies": ["."], "graphs": graphs}), encoding="utf-8")
    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    if keyless:
        for key in list(env):
            if key.startswith(("LANGSMITH_", "LANGCHAIN_")):
                env.pop(key, None)
        env["LANGSMITH_TRACING"] = "false"
    log = folder / "server.tmp"
    handle = log.open("w", encoding="utf-8")
    process = subprocess.Popen(["uv", "run", "langgraph", "dev", "--port", str(port),
                                "--config", str(config), "--no-browser", "--no-reload",
                                "--allow-blocking"],
                               cwd=folder, env=env, stdout=handle, stderr=subprocess.STDOUT,
                               creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
                               start_new_session=os.name != "nt")
    servers.append((process, log, handle))
    url = f"http://127.0.0.1:{port}"
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError("Agent Server exited. See " + str(log))
        try:
            response = httpx.get(url + "/ok", timeout=3, trust_env=False)
            if response.status_code == 200:
                print("server :", url, "ready")
                return url
        except httpx.TransportError:
            pass
        time.sleep(0.5)
    stop_servers()
    raise TimeoutError("Agent Server readiness timed out.")


def server_facts(index=0):
    text = servers[index][1].read_text(encoding="utf-8", errors="replace")
    studio = ""
    for line in text.splitlines():
        if "api_key" in line.lower() or "secret" in line.lower():
            continue
        match = re.search(r"https://[a-z.]*smith\.langchain\.com/studio/", line)
        if match:
            studio = match.group(0)
            break
    print("studio url :", studio)
    print("metadata loop skipped :", "skipping metadata loop" in text)
    print("EU metadata endpoint :", "eu.api.smith.langchain.com" in text)


def assistant_for(url, graph_id):
    response = httpx.post(url + "/assistants/search", json={"limit": 50}, timeout=30, trust_env=False)
    response.raise_for_status()
    matches = [row["assistant_id"] for row in response.json() if row.get("graph_id") == graph_id]
    if len(matches) != 1:
        raise RuntimeError("Expected one assistant for " + graph_id)
    derived = str(uuid5(NAMESPACE_GRAPH, graph_id))
    print("graph id :", graph_id)
    print("assistant id :", matches[0])
    print("uuid5 match :", matches[0] == derived)
    return matches[0]


def rpc(url, assistant, method, params):
    request = {"jsonrpc": "2.0", "id": str(uuid4()), "method": method, "params": params}
    response = httpx.post(url + "/a2a/" + assistant, json=request, timeout=180, trust_env=False,
                          headers={"Accept": "application/json"})
    response.raise_for_status()
    body = response.json()
    if "error" in body:
        raise RuntimeError(json.dumps(body["error"]))
    return body["result"]


def send_params(text, context_id=None):
    message = {"role": "ROLE_USER", "messageId": str(uuid4()), "parts": [{"text": text}]}
    if context_id:
        message["contextId"] = context_id
    return {"message": message, "configuration": {"historyScope": "task"}}


def task_text(task):
    return "\n".join(part["text"] for artifact in task.get("artifacts", [])
                     for part in artifact.get("parts", []) if "text" in part)

In [4]:
health_file = NOTEBOOK_DIR / "health_graph.py"
health_file.write_text('from langgraph.graph import StateGraph, MessagesState, START, END\n'
                       'from langchain.messages import AIMessage\n'
                       'b = StateGraph(MessagesState)\n'
                       'b.add_node("reply", lambda s: {"messages": [AIMessage("local echo")]})\n'
                       'b.add_edge(START, "reply")\nb.add_edge("reply", END)\ngraph = b.compile()\n', encoding="utf-8")
try:
    url = start_server(2124, {"echo": "./health_graph.py:graph"}, NOTEBOOK_DIR, keyless=True)
    native = httpx.post(url + "/runs/wait", json={"assistant_id": "echo", "input": {"messages": [
        {"role": "user", "content": "ping"}]}}, timeout=60, trust_env=False)
    native.raise_for_status()
    print("native graph name :", native.json()["messages"][-1]["content"])
    named = httpx.get(url + "/.well-known/agent-card.json", params={"assistant_id": "echo"}, trust_env=False)
    print("graph name card :", named.status_code, named.json()["error"]["message"])
    echo_id = assistant_for(url, "echo")
    rejected = httpx.post(url + "/a2a/echo", json={"jsonrpc": "2.0", "id": "graph-name", "method": "SendMessage",
                          "params": send_params("ping")}, timeout=30, trust_env=False,
                          headers={"Accept": "application/json"})
    print("graph name rpc :", rejected.json()["error"]["message"])
    echo = rpc(url, echo_id, "SendMessage", send_params("Check local execution."))["task"]
    print("without LangSmith key :", echo["status"]["state"], task_text(echo))
    server_facts()
finally:
    stop_servers()

server : http://127.0.0.1:2124 ready


native graph name : local echo
graph name card : 400 Failed to get assistant 'echo': Invalid assistant ID: must be a UUID
graph id : echo
assistant id : 046b5dc7-2108-582c-9c6c-31d556d138f0
uuid5 match : True
graph name rpc : Invalid assistant ID: must be a UUID


without LangSmith key : TASK_STATE_COMPLETED local echo
studio url : https://smith.langchain.com/studio/
metadata loop skipped : True
EU metadata endpoint : False


We load the course model from the shared setup source in the served graph. The subprocess inherits the repository environment, including the course request budget. We call the native runs/wait endpoint before using A2A.

In [5]:
(NOTEBOOK_DIR / "server_model.py").write_text('\nimport runpy\nfrom pathlib import Path\n\nROOT = next(p for p in Path(__file__).resolve().parents if (p / "pyproject.toml").exists())\nsetup = runpy.run_path(str(ROOT / "build" / "setup_cell.py"))\nllm = setup["make_llm"](max_tokens=500, max_retries=0)\n', encoding="utf-8")
(NOTEBOOK_DIR / "parts_graph.py").write_text('\nimport json\nimport sqlite3\nfrom langchain.messages import SystemMessage\nfrom langgraph.graph import END, START, MessagesState, StateGraph\nfrom server_model import ROOT, llm\n\n\nclass ServedState(MessagesState):\n    usage: dict\n\n\ndef answer(state):\n    with sqlite3.connect(f"file:{(ROOT / \'data/demo-world/service.db\').as_posix()}?mode=ro", uri=True) as db:\n        rows = db.execute("SELECT part_no, stock, depot FROM parts WHERE part_no IN (\'CE-CF-02\', \'CE-DA-03\')").fetchall()\n    reply = llm.invoke([SystemMessage("Answer using this inventory only. Retain every count and depot.\\n" + json.dumps(rows)),\n                        *state["messages"]])\n    return {"messages": [reply], "usage": reply.usage_metadata or {}}\n\n\nbuilder = StateGraph(ServedState)\nbuilder.add_node("answer", answer)\nbuilder.add_edge(START, "answer")\nbuilder.add_edge("answer", END)\ngraph = builder.compile()\n', encoding="utf-8")
url = start_server(2124, {"parts": "./parts_graph.py:graph"}, NOTEBOOK_DIR)
try:
    response = httpx.post(url + "/runs/wait", json={"assistant_id": "parts", "input": {"messages": [
        {"role": "user", "content": "Report CE-CF-02 stock and its depot."}]}}, timeout=180, trust_env=False)
    response.raise_for_status()
    served = response.json()
    print("native answer :", served["messages"][-1]["content"])
    print("usage :", served["usage"])
except Exception:
    stop_servers()
    raise

server : http://127.0.0.1:2124 ready


native answer : CE-CF-02: 6 in stock at Rietpoort.
usage : {'input_tokens': 65, 'output_tokens': 19, 'total_tokens': 84, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## b · agent-card

We fetch the generated agent card and use its advertised JSON-RPC interface. The card describes the assistant and its transport; it grants no credentials or permissions.

In [6]:
try:
    parts_id = assistant_for(url, "parts")
    card_response = httpx.get(url + "/.well-known/agent-card.json", params={"assistant_id": parts_id}, trust_env=False)
    card_response.raise_for_status()
    card = card_response.json()
    print("agent card :", json.dumps(card, indent=2))
    print("card keys :", sorted(card))
    print("security on card :", {key: key in card for key in ["securitySchemes", "security", "signatures"]})
    task = rpc(url, parts_id, "SendMessage", send_params("Report CE-DA-03 stock and its depot."))["task"]
    print("a2a state :", task["status"]["state"])
    print("a2a answer :", task_text(task))
    print("task keys :", sorted(task))
    context = task["contextId"]
    same = rpc(url, parts_id, "GetTask", {"id": task["id"], "historyScope": "task"})
    listed = rpc(url, parts_id, "ListTasks", {"contextId": context, "historyScope": "task"})
    print("get keys :", sorted(same))
    print("get status :", same["status"]["state"])
    print("get status text :", same["status"]["message"]["parts"][0]["text"])
    print("list count :", len(listed["tasks"]))
    short = rpc(url, parts_id, "GetTask", {"id": task["id"], "historyLength": 1})
    print("A2A historyLength 1 :", len(short.get("history", [])), "of", len(same.get("history", [])))
    for method in ["SubscribeToTask", "CreateTaskPushNotificationConfig"]:
        probe = {"jsonrpc": "2.0", "id": method, "method": method, "params": {"id": task["id"]}}
        error = httpx.post(url + "/a2a/" + parts_id, json=probe, trust_env=False,
                           headers={"Accept": "application/json"}).json()["error"]
        print("unsupported :", method, error["code"], error["message"])
    native_state = httpx.get(url + "/threads/" + context + "/state", trust_env=False)
    native_state.raise_for_status()
    print("context is thread :", native_state.json()["values"]["messages"][-1]["content"])
    invalid = {"jsonrpc": "2.0", "id": "invalid-context", "method": "SendMessage",
               "params": send_params("Check inventory.", "BW-7001")}
    print("invalid context :", httpx.post(url + "/a2a/" + parts_id, json=invalid, trust_env=False,
                                           headers={"Accept": "application/json"}).json()["error"]["message"])
    server_facts()
except Exception:
    stop_servers()
    raise

graph id : parts
assistant id : b67dba3b-3342-5f55-b1fe-bdcc570cc1a4
uuid5 match : True
agent card : {
  "name": "parts",
  "description": "parts assistant",
  "url": "http://127.0.0.1:2124/a2a/b67dba3b-3342-5f55-b1fe-bdcc570cc1a4",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2124/a2a/b67dba3b-3342-5f55-b1fe-bdcc570cc1a4",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "stateTransitionHistory": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "required": false,
        "params": {
          "header": "LangGraph-A2A-History-Scope",
          "values": [
            "context",
            "task"
          ],
          "default": "context",
          "methods": [
            "SendMessage",
            "GetTask",
         

a2a state : TASK_STATE_COMPLETED
a2a answer : CE-DA-03: 0 units at Rietpoort.
task keys : ['artifacts', 'contextId', 'history', 'id', 'kind', 'status']
get keys : ['contextId', 'history', 'id', 'kind', 'status']
get status : TASK_STATE_COMPLETED
get status text : Task completed successfully
list count : 1
A2A historyLength 1 : 1 of 2
unsupported : SubscribeToTask -32601 Method not found: SubscribeToTask
unsupported : CreateTaskPushNotificationConfig -32601 Method not found: CreateTaskPushNotificationConfig
context is thread : CE-DA-03: 0 units at Rietpoort.


invalid context : Failed to create run: Invalid thread ID
studio url : https://smith.langchain.com/studio/
metadata loop skipped : False
EU metadata endpoint : True


SendMessage returns result.task, including artifacts with the assistant text. GetTask returns the task directly and, on this server, a generic completion line with no artifacts. On Agent Server, contextId is the server thread id and must be a UUID ; A2A itself treats it as an opaque server value. `historyScope` is an Agent Server option ; the A2A 1.0 field is `historyLength`. The responses still carry the A2A 0.3 field `kind`, and the card has no security scheme. Agent Server implements the JSON-RPC binding only, so the streaming subscription and push-notification methods return -32601.

## c · mcp-tools

We serve one stock tool from an in-memory MCP server and adapt it with `langchain.mcp`. No port and no process : the adapter talks to the FastMCP object directly. An agent then calls the tool like any other.

In [7]:
import mcp.types
from fastmcp import FastMCP
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter

print("MCP protocol revision :", mcp.types.LATEST_PROTOCOL_VERSION)

parts_mcp = FastMCP("veenhaven-parts")


@parts_mcp.tool
def part_stock(part_no: str) -> str:
    """Read the stock count and depot of one Veenhaven Bus part."""
    return json.dumps(stock(part_no))


async with MCPAdapter(parts_mcp) as adapter:
    mcp_tools = await adapter.list_tools()
    print("mcp tools :", [(t.name, t.args) for t in mcp_tools])
    print("direct call :", await mcp_tools[0].ainvoke({"part_no": "CE-CF-02"}))
    stock_agent = create_agent(llm, tools=mcp_tools,
        system_prompt="Answer from part_stock results only. Keep the count and the depot.")
    out = await stock_agent.ainvoke({"messages": [HumanMessage("Report CE-DA-03 stock and its depot.")]}, LIMIT)
print("tool results :", [(m.name, m.content) for m in out["messages"] if isinstance(m, ToolMessage)])
print("answer :", out["messages"][-1].content)

MCP protocol revision : 2026-07-28
mcp tools : [('part_stock', {'part_no': {'type': 'string'}})]


direct call : [{'type': 'text', 'text': '{"part_no": "CE-CF-02", "name": "traction cooling fan", "stock": 6, "price_eur": 420.0, "depot": "Rietpoort"}', 'id': 'lc_af6f8a24-923d-4fee-8f9b-fad98ade4956'}]


tool results : [('part_stock', [{'type': 'text', 'text': '{"part_no": "CE-DA-03", "name": "door actuator", "stock": 0, "price_eur": 890.0, "depot": "Rietpoort"}', 'id': 'lc_2b4a70ff-3cc2-4463-a680-16b9a6ad47f8'}])]
answer : Stock: 0  
Depot: Rietpoort


The tool is async only : `invoke` raises NotImplementedError, so the agent runs with `ainvoke`. The adapter keeps the MCP session open inside the `async with` block. The same Agent Server that answers A2A also publishes every assistant as an MCP tool at `/mcp`. We list the served parts agent through that endpoint without calling it.

In [8]:
try:
    async with MCPAdapter(url + "/mcp") as served_adapter:
        served_tools = await served_adapter.list_tools()
    print("served mcp tools :", [(t.name, sorted(t.args)) for t in served_tools])
except Exception:
    stop_servers()
    raise

served mcp tools : [('parts', ['messages', 'usage'])]


## Things to try

Optional : we can change the part, the context boundary or the policy and inspect the result.

In [9]:
stop_servers()
print("servers stopped :", not servers)
finish_metrics("demo03")

servers stopped : True
metrics : {'model': 'openai/gpt-6-luna', 'rps': 0.4, 'calls': 2, 'input_tokens': 237, 'output_tokens': 37, 'seconds': 24.4}
